## Hard-core sampling w Gibbs sampler

Here is a implementation of Gibbs sampler algorithm for the hard-core problem.

In [1]:
import random
import numpy as np
import pandas as pd

### Create grid for particles

In [12]:
# Set K value for grid size (kxk) suggested 3<=K<=20
K_list = [3, 5, 10, 15, 20]

def create_k_matrix(K):
    # Create a kxk matrix initialized with no particles (you can actually initialize in whichever way you would like lol)
    initial_grid = np.zeros((K, K), dtype=int)
    initial_grid_2 = np.ones((K, K), dtype=int)

    return initial_grid, initial_grid_2

initial_grid, initial_grid_2 = create_k_matrix(K_list[2])

print(f"Created {K_list[2]}x{K_list[2]} grid:")
print(initial_grid)



Created 10x10 grid:
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


### Gibbs random sampler

In [16]:
def run_gibbs_random(grid, K):
    """Move from X_{n} state to X_{n+1} using Gibbs random sampler."""
    # First we select a random vertex on our grid
    i = random.randint(0, K-1)
    j = random.randint(0, K-1)

    # Sample another value for vertex given the other vertex
    # Check if neighbor vertex are 0 (for possible particle in factible grid)
    # Boundary cases
    neighbors_empty = True  # Start assuming neighbors are empty
    
    # Check top neighbor (only if not at top boundary)
    if i > 0 and grid[i-1, j] == 1:
        neighbors_empty = False
    
    # Check bottom neighbor (only if not at bottom boundary)
    if i < K-1 and grid[i+1, j] == 1:
        neighbors_empty = False
    
    # Check left neighbor (only if not at left boundary)
    if j > 0 and grid[i, j-1] == 1:
        neighbors_empty = False
    
    # Check right neighbor (only if not at right boundary)
    if j < K-1 and grid[i, j+1] == 1:
        neighbors_empty = False
    
    if neighbors_empty:
        coin_flipped = random.randint(0,1) # One for heads, zero for tails 
        if coin_flipped == 1:
            grid[i, j] = 1 # Put particle
        else:
            grid[i,j] = 0 # Remove particle
    else:
        grid[i,j] = 0 # If neighbors have particles force no particle in vertex

    return grid
    

### Gibbs sampler w order

In [17]:
def run_gibbs_order(grid, K, i, j):
    """Move from X_{n} state to X{n+1} using Gibbs sampler with order (for faster convergence)."""
    # Check if neighbor vertex are 0 (for possible particle in factible grid)
    # Boundary cases
    neighbors_empty = True  # Start assuming neighbors are empty
    
    # Check top neighbor (only if not at top boundary)
    if i > 0 and grid[i-1, j] == 1:
        neighbors_empty = False
    
    # Check bottom neighbor (only if not at bottom boundary)
    if i < K-1 and grid[i+1, j] == 1:
        neighbors_empty = False
    
    # Check left neighbor (only if not at left boundary)
    if j > 0 and grid[i, j-1] == 1:
        neighbors_empty = False
    
    # Check right neighbor (only if not at right boundary)
    if j < K-1 and grid[i, j+1] == 1:
        neighbors_empty = False
    
    if neighbors_empty:
        coin_flipped = random.randint(0,1) # One for heads, zero for tails 
        if coin_flipped == 1:
            grid[i, j] = 1 # Put particle
        else:
            grid[i,j] = 0 # Remove particle
    else:
        grid[i,j] = 0 # If neighbors have particles force no particle in vertex
    # We pass every vertex of grid in order
    if i == K-1:
        if j == K-1:
            # If we changed last column, last row, start in the beginning of the grid
            i = 0
            j = 0
        else:
            j += 1
    else:
        if j == K-1:
            # If we changed last column, i row, change row and j = 0
            i += 1
            j = 0
        else:
            j += 1
    
    return grid, i, j


### Run Gibbs sampler

In [28]:
final_time = 100001
chain_times = [500, 5000, 10000, 10001, 10002, 100000]

# Run Gibbs random sampler
def get_gibbs_samples_random(chain_times):
    final_grids = []
    for k in K_list:
        print(f"For K={k} we have:")
        updated_grid, updated_grid_2 = create_k_matrix(k)
        for t in range(final_time):
            updated_grid = run_gibbs_random(updated_grid, k)
            if t in chain_times:
                final_grids.append(updated_grid)
                print(f"X_{t} grid:\n{updated_grid}")
    
    return final_grids

get_gibbs_samples_random(chain_times)
            

For K=3 we have:
X_500 grid:
[[0 0 1]
 [0 1 0]
 [1 0 1]]
X_5000 grid:
[[0 0 0]
 [1 0 0]
 [0 0 1]]
X_10000 grid:
[[0 1 0]
 [1 0 0]
 [0 0 0]]
X_10001 grid:
[[0 1 0]
 [1 0 0]
 [0 0 1]]
X_10002 grid:
[[0 1 0]
 [1 0 0]
 [0 0 1]]
X_100000 grid:
[[0 1 0]
 [0 0 1]
 [1 0 0]]
For K=5 we have:
X_500 grid:
[[1 0 1 0 0]
 [0 0 0 0 1]
 [0 0 0 0 0]
 [1 0 1 0 0]
 [0 0 0 1 0]]
X_5000 grid:
[[0 0 1 0 0]
 [1 0 0 1 0]
 [0 0 1 0 1]
 [1 0 0 1 0]
 [0 1 0 0 0]]
X_10000 grid:
[[0 1 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 0]
 [1 0 0 0 0]]
X_10001 grid:
[[0 1 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 0]
 [1 0 0 0 0]]
X_10002 grid:
[[0 1 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 0]
 [1 0 0 0 0]]
X_100000 grid:
[[1 0 0 0 1]
 [0 0 0 1 0]
 [0 0 0 0 1]
 [0 1 0 0 0]
 [0 0 1 0 0]]
For K=10 we have:
X_500 grid:
[[1 0 0 0 1 0 1 0 0 0]
 [0 0 1 0 0 0 0 0 1 0]
 [0 1 0 0 1 0 0 0 0 1]
 [1 0 0 1 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 1]
 [1 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 1]
 [0 0 0 1 0 0 1 0 0 0]
 [0 1 0 0 0 1 0 0 0 0]
 

[array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0]]),
 array([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0]]),
 array([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0]]),
 array([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0]]),
 array([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0]]),
 array([[1,

In [26]:
# Run gibbs w order sampler
chain_times = [500, 5000, 10000, 10001, 10002, 100000]
i = 0
j = 0
def get_gibbs_samples_order(chain_times, i, j):
    final_grids = []
    for k in K_list:
        print(f"For K={k} we have:")
        updated_grid, updated_grid_2 = create_k_matrix(k)
        for t in range(final_time):
            updated_grid, i, j = run_gibbs_order(updated_grid, k, i, j)
            if t in chain_times:
                final_grids.append(updated_grid)
                print(f"X_{t} grid:\n{updated_grid}")

    return final_grids

get_gibbs_samples_order(chain_times, i, j)
        

For K=3 we have:
X_500 grid:
[[0 0 0]
 [0 0 0]
 [0 0 1]]
X_5000 grid:
[[0 0 0]
 [0 1 0]
 [0 0 1]]
X_10000 grid:
[[1 0 1]
 [0 1 0]
 [0 0 0]]
X_10001 grid:
[[1 0 1]
 [0 1 0]
 [0 0 0]]
X_10002 grid:
[[1 0 1]
 [0 1 0]
 [0 0 0]]
For K=5 we have:
X_500 grid:
[[1 0 0 1 0]
 [0 1 0 0 1]
 [1 0 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
X_5000 grid:
[[1 0 1 0 0]
 [0 1 0 0 0]
 [1 0 0 0 1]
 [0 1 0 1 0]
 [0 0 1 0 1]]
X_10000 grid:
[[0 1 0 0 1]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 1]
 [0 1 0 0 0]]
X_10001 grid:
[[0 1 0 0 1]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 1]
 [0 1 0 0 0]]
X_10002 grid:
[[0 1 0 0 1]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 1]
 [0 1 0 0 0]]
For K=10 we have:
X_500 grid:
[[1 0 0 0 1 0 1 0 0 1]
 [0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 1 0 0 1 0 0]
 [0 1 0 1 0 0 0 0 0 0]
 [0 0 1 0 1 0 1 0 0 0]
 [0 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 1 0 1]
 [0 0 1 0 0 0 0 0 0 0]
 [1 0 0 1 0 1 0 0 0 1]]
X_5000 grid:
[[1 0 0 0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0 1 0 1]
 [0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 1 0

[array([[1, 0, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 0],
        [0, 0, 1],
        [1, 0, 0]]),
 array([[1, 0, 1, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0]]),
 array([[1, 0, 1, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0]]),
 array([[1, 0, 1, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0]]),
 array([[1, 0, 1, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0]]),
 array([[1, 0, 1, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0]]),
 array([[0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 0, 1, 

## Hard-core particle estimation

Using the samples made in the past code we will make a histogram of number of particles for different chain times.